# Project 2: Transformers

This project is part of the NLP module held in the spring of 2026. Three transformer models are compared, answering physical common sense tasks with the PIQA dataset. 

- **Randomly Initialized Transformer** 
- **Pretrained Transformer** not pretrained or finetuned on PIQA dataset
- **LLM (1B+ Parameters)** same hyperparameters

**Dataset**  
"PIQA: Reasoning about Physical Commonsense in Natural Language" — a binary choice task
where a model selects the more physically plausible solution to a given goal.  
Source: [https://arxiv.org/abs/1911.11641](https://arxiv.org/abs/1911.11641)

**Tools** 
- Course Materials
- Documentations (mostly of imported dependencies)
    - apxml
    - NLTK
    - PyTorch
    - skikit-learn
- Claude AI for the following tasks:
    - Helping formulate and clarify reasoning
    - General coding assistance
- Regex101
- Huggingface

**Weights & Biases**  
All experimental runs are logged and published in the
# TODO report

**Notebook structure**
1. Introduction
2. Setup
3. Preprocessing
4. Model
5. Training
6. Evaluation
7. Interpretation


## Setup

In [1]:
!pip install \
    datasets==4.8.4 \
    numpy==2.4.4 \
    datetime==6.0.0 \
    transformers==5.6.2 \
    wandb==0.25.1


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from datasets import load_dataset
from datetime import datetime
from transformers import AutoTokenizer
import numpy as np
import wandb
import re

In [3]:
SEED = 42
np.random.seed(SEED)

In [4]:
TS = datetime.now().strftime("%Y%m%d_%H%M%S")

In [5]:
wandb_project = "nlp-project2-piqa"

## Preprocessing

In [6]:
train_split = load_dataset("ybisk/piqa", split="train[:-1000]", revision='refs/convert/parquet')
valid_split = load_dataset("ybisk/piqa", split="train[-1000:]", revision='refs/convert/parquet')
test_split = load_dataset("ybisk/piqa", split="validation", revision='refs/convert/parquet')

### Feature Selection

In [7]:
COL_GOAL = 'goal'
COL_SOL1 = 'sol1'
COL_SOL2 = 'sol2'
COL_LABEL = 'label'

### Filter HTML Elements

In [8]:
# regex source: https://apxml.com/courses/nlp-fundamentals/chapter-1-nlp-text-processing-techniques/handling-text-noise
# verified with: https://regex101.com
regex_pattern = re.compile(r'<[^>]+>', re.IGNORECASE)
html_elements = 0

for split in [train_split, valid_split, test_split]:
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_GOAL])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL1])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL2])))

print(f"Number of HTML elements found: {html_elements}")

Number of HTML elements found: 0


### Input Format

In [9]:
MAX_INPUT_LENGTH = 553 # value found by analysis further down

COL_INPUT1 = 'input1'
COL_INPUT2 = 'input2'

ATTENTION_MASK = 'attention_mask'
INPUT_IDS = 'input_ids'
TOKEN_TYPE_IDS = 'token_type_ids'

COL_INPUT1_ATTENTION_MASK  = f"{COL_INPUT1}_{ATTENTION_MASK}"
COL_INPUT1_INPUT_IDS       = f"{COL_INPUT1}_{INPUT_IDS}"
COL_INPUT1_TOKEN_TYPE_IDS  = f"{COL_INPUT1}_{TOKEN_TYPE_IDS}"
COL_INPUT2_ATTENTION_MASK  = f"{COL_INPUT2}_{ATTENTION_MASK}"
COL_INPUT2_INPUT_IDS       = f"{COL_INPUT2}_{INPUT_IDS}"
COL_INPUT2_TOKEN_TYPE_IDS  = f"{COL_INPUT2}_{TOKEN_TYPE_IDS}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_row(row):
    tokenized1 = tokenizer(row[COL_GOAL], row[COL_SOL1], truncation=True, max_length=MAX_INPUT_LENGTH)
    tokenized2 = tokenizer(row[COL_GOAL], row[COL_SOL2], truncation=True, max_length=MAX_INPUT_LENGTH)
    return {
        COL_LABEL: row[COL_LABEL],
        COL_INPUT1_ATTENTION_MASK: tokenized1[ATTENTION_MASK],
        COL_INPUT1_INPUT_IDS: tokenized1[INPUT_IDS],
        COL_INPUT1_TOKEN_TYPE_IDS: tokenized1[TOKEN_TYPE_IDS],
        COL_INPUT2_ATTENTION_MASK: tokenized2[ATTENTION_MASK],
        COL_INPUT2_INPUT_IDS: tokenized2[INPUT_IDS],
        COL_INPUT2_TOKEN_TYPE_IDS: tokenized2[TOKEN_TYPE_IDS],
    }

train_processed = train_split.map(preprocess_row, remove_columns=train_split.column_names, batched=False)
valid_processed = valid_split.map(preprocess_row, remove_columns=valid_split.column_names, batched=False)
test_processed = test_split.map(preprocess_row,  remove_columns=test_split.column_names, batched=False)

print(train_processed[0])

Map:   0%|          | 0/15113 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1838 [00:00<?, ? examples/s]

{'label': 1, 'input1_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'input1_input_ids': [101, 2043, 16018, 12136, 1010, 2043, 2009, 1005, 1055, 3201, 1010, 2017, 2064, 102, 10364, 2009, 3031, 1037, 5127, 102], 'input1_token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], 'input2_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'input2_input_ids': [101, 2043, 16018, 12136, 1010, 2043, 2009, 1005, 1055, 3201, 1010, 2017, 2064, 102, 10364, 2009, 2046, 1037, 15723, 102], 'input2_token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]}


### Length Analysis

In [10]:
all_input_lengths = (
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in train_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in train_processed] +
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in valid_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in valid_processed] +
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in test_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in test_processed]
)

print(f"Max length of an input field: {np.max(all_input_lengths)}") 

Max length of an input field: 553


## Model

### Randomly Initialized Transformer

### Pretrained Transformer

### LLM (1B+ Parameters)

## Training

In [11]:
SKIP_TRAINING = True
MODEL1_NAME = f"random_{TS}"
MODEL2_NAME = f"pretrained_{TS}"
SWEEP_COUNT = 5


In [12]:
training_config = {
    'max_epochs': 30,
    'patience': 5,
}

# use same model architecture in model 1 and 2
# todo config
def create_sweep_config(model_name):
    return {
        "method": "bayes",
        "metric": {"name": f"{model_name}/valid_acc", "goal": "maximize"},
        "parameters": {
            "lr":                  {"values": [1e-3, 1e-4, 1e-5]},
            "weight_decay":        {"values": [1e-3, 1e-4, 1e-5]},
        },
    }

In [13]:
if not SKIP_TRAINING:
    wandb.login()
else: 
    print("training skipped")

training skipped


In [14]:
def train_model(model, config):
    print(f"model: {model} and config: {config}")

In [15]:
def sweep_run_model1():
    with wandb.init(project=wandb_project, config=training_config, group='random') as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL1_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = {}
        
        train_model(model, config)
        
        
if not SKIP_TRAINING:
    sweep_model1 = wandb.sweep(sweep=create_sweep_config(MODEL1_NAME), project=wandb_project)
    
    wandb.agent(sweep_model1, function=sweep_run_model1, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


In [16]:
def sweep_run_model2():
    with wandb.init(project=wandb_project, config=training_config, group='pretrained') as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL2_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = {}
        
        train_model(model, config)
        

if not SKIP_TRAINING:
    sweep_model2 = wandb.sweep(sweep=create_sweep_config(MODEL2_NAME), project=wandb_project)
    
    wandb.agent(sweep_model2, function=sweep_run_model2, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


## Evaluation

## Interpretation